# 17 - Process production videos with executor partitions

Processes one explicitly routed `EXECUTOR_PARTITION` claim in bounded Spark waves. Executors only produce immutable attempt-scoped staging records. The driver synchronously heartbeats and independently publishes every attempt through the production worker-event and commit contract.

The first code cell is the Fabric startup configuration. Attach the same Lakehouse and Environment as notebooks 03 and 04, stop an existing session after startup-setting changes, and mark the following Python cell as the parameter cell.

In [ ]:
%%configure
{
  "driverMemory": {
    "parameterName": "DRIVER_MEMORY",
    "defaultValue": "28g"
  },
  "driverCores": {
    "parameterName": "DRIVER_CORES",
    "defaultValue": 4
  },
  "executorMemory": {
    "parameterName": "EXECUTOR_MEMORY",
    "defaultValue": "28g"
  },
  "executorCores": {
    "parameterName": "EXECUTOR_CORES",
    "defaultValue": 16
  },
  "conf": {
    "spark.task.cpus": {
      "parameterName": "CPUS_PER_TASK",
      "defaultValue": "1"
    },
    "spark.speculation": "false",
    "spark.scheduler.mode": "FIFO",
    "spark.dynamicAllocation.enabled": "true",
    "spark.dynamicAllocation.minExecutors": {
      "parameterName": "MIN_EXECUTORS",
      "defaultValue": "1"
    },
    "spark.dynamicAllocation.maxExecutors": {
      "parameterName": "MAX_EXECUTORS",
      "defaultValue": "4"
    }
  }
}


In [ ]:
WORK_ITEMS_JSON = "[]"
WORK_ID = ""
ATTEMPT_ID = ""
MAX_ITEMS_PER_WORKER = 1
MAX_WORKER_LIFETIME_SECONDS = 19800
PIPELINE_RUN_ID = ""
ACTIVITY_RUN_ID = ""
WORKER_EXECUTION_ID = ""
FABRIC_JOB_INSTANCE_ID = ""
BUNDLE_MANIFEST_SHA256 = ""
MODELS_DIR = "/lakehouse/default/Files/models/<release>"
SOURCE_STORAGE_ACCOUNT = ""
SOURCE_CONTAINER = ""
SOURCE_SHORTCUT_LOCAL_ROOT = "/lakehouse/default/Files/videos"
DATABASE = ""
TABLE_PREFIX = "people_counter"
LEASE_MINUTES = 360
HEARTBEAT_SECONDS = 600
PROCESSING_ENGINE = "EXECUTOR_PARTITION"
CPUS_PER_TASK = 1
PARALLEL_TASKS = "auto"
PARTITION_WAVES = 1
PEAK_WORKER_MEMORY_GIB = 12.0
RESOURCE_DISCOVERY_TIMEOUT_SECONDS = 120
MIN_APPROVED_SINGLE_VIDEO_SPEED_X = 1.0
LEASE_SAFETY_FACTOR = 1.5
LEASE_SAFETY_MARGIN_SECONDS = 1200
DRIVER_MEMORY = "28g"
DRIVER_CORES = 4
EXECUTOR_MEMORY = "28g"
EXECUTOR_CORES = 16
MIN_EXECUTORS = 1
MAX_EXECUTORS = 4

In [ ]:
from datetime import datetime, timedelta, timezone
from importlib.metadata import version
from pathlib import Path, PurePosixPath
from urllib.parse import unquote, urlsplit
import json
import math
import re
import time

import notebookutils
from people_counter import RFDetrBotsortConfig, RTDetrOsnetConfig
from people_counter.cpu_runtime import calculate_placement_safe_thread_budget, configure_cpu_runtime
from people_counter.fabric_control import ControlWriter
from people_counter.fabric_dispatch import EXECUTOR_PARTITION, normalize_processing_engine
from people_counter.fabric_events import LeaseLostError, WorkerEventClient
from people_counter.fabric_executor_production import (
    DriverAttemptStates,
    ExecutorPreflightError,
    LeaseSafetyControls,
    StagingConflictError,
    StagingTransaction,
    parse_claimed_items,
    plan_largest_cost_first,
    process_production_partition,
    require_configured_lease_budget,
    require_live_wave_budget,
    staging_transaction,
    validate_claimed_rows,
    validate_staging_records,
)
from people_counter.runtime import RuntimeCompatibilityError
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, StringType, StructField, StructType


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def required_text(value, name):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{name} must be a non-empty string")
    return value.strip()


def table(suffix):
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


def utc_now():
    return datetime.now(timezone.utc)


def memory_gib(value):
    text = required_text(str(value), "memory").lower()
    match = re.fullmatch(r"([0-9]+(?:\.[0-9]+)?)([gmk])", text)
    if not match:
        raise ValueError(f"Unsupported Spark memory value: {value!r}")
    amount = float(match.group(1))
    return amount * {"g": 1.0, "m": 1 / 1024, "k": 1 / (1024 * 1024)}[match.group(2)]


def source_path(work):
    uri = required_text(work["source_uri"], "source_uri")
    parts = urlsplit(uri)
    account = required_text(SOURCE_STORAGE_ACCOUNT, "SOURCE_STORAGE_ACCOUNT").lower()
    container = required_text(SOURCE_CONTAINER, "SOURCE_CONTAINER")
    expected_netloc = f"{container}@{account}.dfs.core.windows.net"
    if parts.scheme.lower() not in {"abfs", "abfss"} or parts.netloc.lower() != expected_netloc:
        raise ValueError("source_uri is outside the reviewed source namespace")
    decoded = unquote(parts.path)
    relative = PurePosixPath(decoded.lstrip("/"))
    if not relative.parts or any(part in {"", ".", ".."} for part in relative.parts):
        raise ValueError("source_uri contains an unsafe path")
    root = Path(required_text(SOURCE_SHORTCUT_LOCAL_ROOT, "SOURCE_SHORTCUT_LOCAL_ROOT")).resolve()
    candidate = root.joinpath(*relative.parts).resolve()
    if root not in candidate.parents:
        raise ValueError("source_uri escapes SOURCE_SHORTCUT_LOCAL_ROOT")
    return candidate


def build_config(work, staged):
    settings = json.loads(work["config_json"])
    common = {
        "video": staged,
        "device_variant": settings["device_variant"],
        "device": settings["device"],
        "batch_size": int(settings["batch_size"]),
        "sample_fps": settings["sample_fps"],
        "detection_threshold": float(settings["detection_threshold"]),
        "use_fp16": bool(settings["use_fp16"]),
        "line": tuple(settings["line"]) if settings["line"] else None,
        "models_dir": Path(MODELS_DIR),
    }
    if settings["pipeline"] == "rtdetr-osnet":
        return RTDetrOsnetConfig(**common, detector_model=settings["detector_model"])
    if settings["pipeline"] == "rfdetr-botsort":
        return RFDetrBotsortConfig(
            **common,
            camera_motion_compensation=settings["camera_motion_compensation"],
        )
    raise ValueError(f"Unsupported pipeline: {settings['pipeline']}")


def classify_error(error):
    if isinstance(error, LeaseLostError):
        return True, "LEASE"
    if isinstance(error, FileNotFoundError):
        return True, "STORAGE"
    if isinstance(error, (ValueError, PermissionError, RuntimeCompatibilityError)):
        return False, "INPUT"
    return True, "RUNTIME"


def attempt_updates(status, extra=None):
    return {
        **(extra or {}),
        "status": status,
        "last_heartbeat_at": utc_now(),
        "pipeline_run_id": PIPELINE_RUN_ID or None,
        "activity_run_id": ACTIVITY_RUN_ID or None,
        "fabric_job_instance_id": FABRIC_JOB_INSTANCE_ID or None,
        "sdk_version": version("people-counter"),
        "bundle_manifest_sha256": BUNDLE_MANIFEST_SHA256 or None,
    }


def heartbeat_context(context, status, extra=None):
    context["client"].submit(
        "heartbeat",
        {"status": status, "updates": attempt_updates(status, extra)},
    )


def append_attempt_rows(table_name, frame, attempt_id):
    if frame.take(1):
        (
            frame.write.format("delta").mode("append")
            .option("txnAppId", f"{table_name}:{attempt_id}")
            .option("txnVersion", 0)
            .saveAsTable(table_name)
        )


def publication_frames(work, attempt_id, records):
    recorded_at = utc_now()
    common = {
        "work_id": work["work_id"],
        "attempt_id": attempt_id,
        "camera_id": work["camera_id"],
        "location_id": work["location_id"],
        "captured_at_utc": work["captured_at_utc"],
        "capture_date": work["capture_date"],
        "recorded_at": recorded_at,
    }
    telemetry = []
    line_counts = []
    for record in records:
        payload = json.loads(record["payload_json"])
        if record["record_type"] == "telemetry":
            telemetry.append({
                **common,
                **payload,
                "person_entry_at_utc": work["captured_at_utc"] + timedelta(seconds=payload["entry_seconds"]),
                "person_exit_at_utc": work["captured_at_utc"] + timedelta(seconds=payload["exit_seconds"]),
            })
        elif record["record_type"] == "line_count":
            line_counts.append({
                **common,
                **payload,
                "observed_at_utc": work["captured_at_utc"] + timedelta(seconds=float(payload["video_seconds"])),
            })
    return (
        spark_session.createDataFrame(telemetry, spark_session.table(telemetry_table).schema),
        spark_session.createDataFrame(line_counts, spark_session.table(line_counts_table).schema),
    )


In [ ]:
worker_started = time.monotonic()
worker_items = parse_claimed_items(WORK_ITEMS_JSON, MAX_ITEMS_PER_WORKER)
if not worker_items:
    outcome = {
        "processing_engine": EXECUTOR_PARTITION,
        "worker_execution_id": WORKER_EXECUTION_ID or None,
        "status": "NO_WORK",
        "claimed_count": 0,
        "succeeded_count": 0,
        "failed_count": 0,
        "retryable_count": 0,
    }
    notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))

if normalize_processing_engine(PROCESSING_ENGINE, default=None) != EXECUTOR_PARTITION:
    raise ValueError("PROCESSING_ENGINE must equal EXECUTOR_PARTITION")
worker_execution_id = required_text(WORKER_EXECUTION_ID, "WORKER_EXECUTION_ID")
dispatcher_id = required_text(PIPELINE_RUN_ID, "PIPELINE_RUN_ID")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")
lease_minutes = int(LEASE_MINUTES)
heartbeat_seconds = int(HEARTBEAT_SECONDS)
partition_waves = int(PARTITION_WAVES)
cpus_per_task = int(CPUS_PER_TASK)
peak_worker_memory_gib = float(PEAK_WORKER_MEMORY_GIB)
if lease_minutes < 5 or heartbeat_seconds < 30 or partition_waves < 1:
    raise ValueError("Lease, heartbeat, and partition-wave controls are invalid")
if cpus_per_task < 1 or peak_worker_memory_gib <= 0 or not math.isfinite(peak_worker_memory_gib):
    raise ValueError("Executor CPU and memory controls must be positive")
controls = LeaseSafetyControls.create(
    MIN_APPROVED_SINGLE_VIDEO_SPEED_X,
    LEASE_SAFETY_FACTOR,
    LEASE_SAFETY_MARGIN_SECONDS,
    heartbeat_seconds,
)

spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
spark_configuration = spark_session.sparkContext.getConf()
effective_task_cpus = int(spark_configuration.get("spark.task.cpus", "1"))
if effective_task_cpus != cpus_per_task:
    raise RuntimeError(
        f"CPUS_PER_TASK requested {cpus_per_task} but Spark uses {effective_task_cpus}; stop the session and restart with the reviewed profile"
    )
effective_executor_cores = int(
    spark_configuration.get("spark.executor.cores", str(EXECUTOR_CORES))
)
if effective_executor_cores != int(EXECUTOR_CORES):
    raise RuntimeError(
        f"EXECUTOR_CORES requested {EXECUTOR_CORES} but Spark uses {effective_executor_cores}; stop the session and restart with the reviewed profile"
    )
if spark_configuration.get("spark.speculation", "false").lower() != "false":
    raise RuntimeError("spark.speculation must be false for stable staging transactions")
control_writer = ControlWriter(spark_session, table("control_writer"))
work_table = table("video_work")
attempts_table = table("video_attempts")
staging_table = table("executor_attempt_results")
telemetry_table = table("telemetry_attempts")
line_counts_table = table("line_count_attempts")
claimed_work_ids = [item["work_id"] for item in worker_items]
queue_rows = [
    row.asDict(recursive=True)
    for row in spark_session.table(work_table).where(F.col("work_id").isin(claimed_work_ids)).collect()
]
work_rows = validate_claimed_rows(worker_items, queue_rows, dispatcher_id=dispatcher_id)
for row in work_rows:
    row["attempt_id"] = row["lease_owner_attempt_id"]
projections = require_configured_lease_budget(work_rows, controls, lease_minutes)

states = DriverAttemptStates(item["attempt_id"] for item in worker_items)
contexts = {}
for work in work_rows:
    client = WorkerEventClient(
        spark_session,
        control_writer,
        table_prefix=prefix,
        database=database,
        work_id=work["work_id"],
        attempt_id=work["attempt_id"],
        worker_execution_id=worker_execution_id,
        capture_date=work["capture_date"],
        lease_minutes=lease_minutes,
    )
    client.submit("claim_execution", {})
    context = {"work": work, "client": client}
    contexts[work["attempt_id"]] = context
    heartbeat_context(context, "LEASED")

discovery_deadline = time.monotonic() + int(RESOURCE_DISCOVERY_TIMEOUT_SECONDS)
observed_executor_count = 0
while observed_executor_count < 1 and time.monotonic() < discovery_deadline:
    executor_list = spark_session.sparkContext._jsc.sc().statusStore().executorList(True)
    observed_executor_count = max(0, int(executor_list.size()) - 1)
    if observed_executor_count < 1:
        time.sleep(1)
if observed_executor_count < 1:
    raise ExecutorPreflightError("No active Spark executor was discovered before the timeout")
executor_cores = int(EXECUTOR_CORES)
scheduler_cap = observed_executor_count * (executor_cores // cpus_per_task)
memory_cap = observed_executor_count * int(memory_gib(EXECUTOR_MEMORY) // peak_worker_memory_gib)
explicit_cap = len(work_rows) if str(PARALLEL_TASKS).strip().lower() == "auto" else int(PARALLEL_TASKS)
planned_concurrency = min(len(work_rows), scheduler_cap, memory_cap, explicit_cap)
if planned_concurrency < 1:
    raise ExecutorPreflightError("Observed executor resources cannot place one approved video task")
thread_budget = calculate_placement_safe_thread_budget(
    [executor_cores] * observed_executor_count,
    cpus_per_task,
    planned_concurrency,
)
partitions = plan_largest_cost_first(work_rows, planned_concurrency)
waves = [partitions[index::partition_waves] for index in range(partition_waves)]
waves = [wave for wave in waves if wave]
maximum_renewal_cycle_seconds = 0.0

for wave_index, wave_partitions in enumerate(waves):
    if time.monotonic() - worker_started >= int(MAX_WORKER_LIFETIME_SECONDS):
        raise ExecutorPreflightError("Worker lifetime limit reached before starting a wave")
    wave_rows = [row for partition in wave_partitions for row in partition]
    wave_attempts = {row["attempt_id"] for row in wave_rows}
    for attempt_id in wave_attempts:
        states.transition(attempt_id, "STAGING")
        heartbeat_context(
            contexts[attempt_id],
            "STAGING",
            {"staging_started_at": utc_now()},
        )
    live_rows = [
        row.asDict(recursive=True)
        for row in spark_session.table(work_table).where(F.col("work_id").isin([row["work_id"] for row in wave_rows])).collect()
    ]
    projected_wave_seconds = max(
        controls.projected_wall_seconds(row["duration_seconds"])
        for row in wave_rows
    )
    require_live_wave_budget(
        projected_wave_seconds,
        [row["lease_expires_at"] for row in live_rows],
        margin_seconds=controls.margin_seconds,
    )
    for attempt_id in wave_attempts:
        states.transition(attempt_id, "RUNNING")
        heartbeat_context(
            contexts[attempt_id],
            "RUNNING",
            {"inference_started_at": utc_now()},
        )
    transaction = staging_transaction(worker_execution_id, wave_index)
    prior_rows = [
        row.asDict(recursive=True)
        for row in spark_session.table(staging_table).where(
            (F.col("txn_app_id") == transaction.app_id)
            & (F.col("txn_version") == transaction.version)
        ).collect()
    ]
    wave_items = [item for item in worker_items if item["attempt_id"] in wave_attempts]
    if prior_rows:
        validate_staging_records(prior_rows, wave_items, transaction)
    else:
        planned_rows = []
        for partition_id, partition in enumerate(wave_partitions):
            for row in partition:
                planned_rows.append({**row, "planned_partition": partition_id})
        planned_schema = StructType([
            *spark_session.table(work_table).schema.fields,
            StructField("attempt_id", StringType(), False),
            StructField("planned_partition", IntegerType(), False),
        ])
        planned_frame = spark_session.createDataFrame(planned_rows, planned_schema)
        planned_frame = planned_frame.repartition(len(wave_partitions), "planned_partition")

        def execute_partition(rows):
            configure_cpu_runtime(
                thread_budget.driver_cores,
                thread_budget.active_workers,
            )
            yield from process_production_partition(
                (row.asDict(recursive=True) for row in rows),
                source_resolver=source_path,
                config_builder=build_config,
                worker_execution_id=worker_execution_id,
                transaction=StagingTransaction(transaction.app_id, transaction.version),
                error_classifier=classify_error,
            )

        staged_frame = spark_session.createDataFrame(
            planned_frame.rdd.mapPartitions(execute_partition),
            spark_session.table(staging_table).schema,
        )
        (
            staged_frame.write.format("delta").mode("append")
            .option("txnAppId", transaction.app_id)
            .option("txnVersion", transaction.version)
            .saveAsTable(staging_table)
        )
        materialized_rows = [
            row.asDict(recursive=True)
            for row in spark_session.table(staging_table).where(
                (F.col("txn_app_id") == transaction.app_id)
                & (F.col("txn_version") == transaction.version)
            ).collect()
        ]
        validate_staging_records(materialized_rows, wave_items, transaction)
    renewal_started = time.monotonic()
    for attempt_id, status in states.nonterminal().items():
        heartbeat_context(contexts[attempt_id], status)
    maximum_renewal_cycle_seconds = max(
        maximum_renewal_cycle_seconds,
        time.monotonic() - renewal_started,
    )

staged_rows = [
    row.asDict(recursive=True)
    for row in spark_session.table(staging_table).where(
        F.col("worker_execution_id") == worker_execution_id
    ).collect()
]
item_outcomes = []
control_failures = []
for item in worker_items:
    attempt_id = item["attempt_id"]
    work = contexts[attempt_id]["work"]
    client = contexts[attempt_id]["client"]
    records = [row for row in staged_rows if row["attempt_id"] == attempt_id]
    terminal = [row for row in records if row["record_type"] in {"video_result", "error"}]
    if len(terminal) != 1:
        raise StagingConflictError(f"Attempt {attempt_id} has {len(terminal)} terminal staging rows")
    envelope = terminal[0]
    if envelope["status"] == "FAILED":
        try:
            client.submit("failure", {
                "processed_frames": envelope["processed_frames"],
                "processing_seconds": envelope["processing_seconds"],
                "retryable": bool(envelope["retryable"]),
                "lease_lost": False,
                "error_category": json.loads(envelope["payload_json"])["error_category"],
                "error_type": envelope["error_type"],
                "error_message": envelope["error_message"],
            })
            states.transition(attempt_id, "WRITING")
            states.transition(attempt_id, "RETRY_WAIT" if envelope["retryable"] else "FAILED")
        except Exception as error:
            control_failures.append(f"{attempt_id}: {type(error).__name__}: {error}")
        item_outcomes.append({
            "work_id": item["work_id"],
            "attempt_id": attempt_id,
            "status": "FAILED",
            "retryable": bool(envelope["retryable"]),
        })
        continue
    try:
        states.transition(attempt_id, "WRITING")
        heartbeat_context(
            contexts[attempt_id],
            "WRITING",
            {"writing_started_at": utc_now()},
        )
        telemetry_frame, line_frame = publication_frames(work, attempt_id, records)
        append_attempt_rows(telemetry_table, telemetry_frame, attempt_id)
        append_attempt_rows(line_counts_table, line_frame, attempt_id)
        client.submit("attempt_update", {"updates": {
            "status": "SUCCEEDED",
            "completed_at": utc_now(),
            "input_sha256": envelope["input_sha256"],
            "source_size_bytes": envelope["source_size_bytes"],
            "source_duration_seconds": envelope["source_duration_seconds"],
            "source_fps": envelope["source_fps"],
            "total_source_frames": envelope["total_source_frames"],
            "processed_frames": envelope["processed_frames"],
            "processing_seconds": envelope["processing_seconds"],
            "distinct_people": envelope["distinct_people"],
            "line_in_count": envelope["line_in_count"],
            "line_out_count": envelope["line_out_count"],
            "retryable": None,
            "error_category": None,
            "error_type": None,
            "error_message": None,
        }})
        client.submit("commit", {})
        published = spark_session.table(work_table).where(F.col("work_id") == item["work_id"]).limit(2).collect()
        if len(published) != 1 or published[0].status != "SUCCEEDED" or published[0].committed_attempt_id != attempt_id:
            raise LeaseLostError("Attempt staging was written but the commit pointer was not published")
        states.transition(attempt_id, "SUCCEEDED")
        item_outcomes.append({
            "work_id": item["work_id"],
            "attempt_id": attempt_id,
            "status": "SUCCEEDED",
            "telemetry_rows": telemetry_frame.count(),
            "line_count_rows": line_frame.count(),
        })
    except Exception as error:
        control_failures.append(f"{attempt_id}: {type(error).__name__}: {error}")

if maximum_renewal_cycle_seconds >= heartbeat_seconds / 2:
    control_failures.append(
        f"Heartbeat renewal cycle {maximum_renewal_cycle_seconds:.3f}s exceeds half the interval"
    )
if control_failures:
    raise RuntimeError("Executor control-plane failures: " + "; ".join(control_failures))

wall_seconds = time.monotonic() - worker_started
source_video_minutes = sum(float(row["duration_seconds"]) for row in work_rows) / 60.0
succeeded_count = sum(item["status"] == "SUCCEEDED" for item in item_outcomes)
failed_count = sum(item["status"] == "FAILED" for item in item_outcomes)
retryable_count = sum(item.get("retryable") is True for item in item_outcomes)
outcome = {
    "processing_engine": EXECUTOR_PARTITION,
    "worker_execution_id": worker_execution_id,
    "claimed_count": len(worker_items),
    "succeeded_count": succeeded_count,
    "failed_count": failed_count,
    "retryable_count": retryable_count,
    "observed_executor_count": observed_executor_count,
    "planned_concurrency": planned_concurrency,
    "observed_concurrency": planned_concurrency,
    "native_threads_per_worker": thread_budget.threads_per_worker,
    "maximum_heartbeat_cycle_seconds": maximum_renewal_cycle_seconds,
    "wall_seconds": wall_seconds,
    "source_video_minutes": source_video_minutes,
    "throughput_video_minutes_per_wall_minute": (
        source_video_minutes / (wall_seconds / 60.0) if wall_seconds > 0 else None
    ),
    "items": item_outcomes,
}
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))